In [18]:
import os
import json
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

In [19]:
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

In [20]:
with open("/content/records_long.json", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)


In [21]:
df = df[["text", "model"]].copy()
df = df.dropna(subset=["text", "model"])
df["text"] = df["text"].astype(str)

print(df.shape)
print(df["model"].value_counts())

(320593, 2)
model
llama      86120
mistral    81448
gemma      69388
human      49050
chatgpt    34587
Name: count, dtype: int64


In [22]:
# Keep the same label mapping used in your final notebook
label_mapping = {
    "chatgpt": "OpenAI",
    "gemma": "Google",
    "human": "Human",
    "llama": "Meta",
    "mistral": "Anthropic"
}

df["final_label"] = df["model"].map(label_mapping)
df = df.dropna(subset=["final_label"])

print(df["final_label"].value_counts())
print(df.shape)

final_label
Meta         86120
Anthropic    81448
Google       69388
Human        49050
OpenAI       34587
Name: count, dtype: int64
(320593, 3)


In [23]:
USE_SAMPLE = True
SAMPLE_FRAC = 0.3   # only used if USE_SAMPLE = True

if USE_SAMPLE:
    df = (
        df.groupby("model", group_keys=False)
          .apply(lambda x: x.sample(frac=SAMPLE_FRAC, random_state=42))
          .reset_index(drop=True)
    )

print(df.shape)
print(df["final_label"].value_counts())

(96177, 3)
final_label
Meta         25836
Anthropic    24434
Google       20816
Human        14715
OpenAI       10376
Name: count, dtype: int64


/tmp/ipykernel_1339/3912021403.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=SAMPLE_FRAC, random_state=42))


In [24]:
encoder = LabelEncoder()
df["label_id"] = encoder.fit_transform(df["final_label"])

print("Class mapping:")
for i, class_name in enumerate(encoder.classes_):
    print(f"{i} -> {class_name}")

Class mapping:
0 -> Anthropic
1 -> Google
2 -> Human
3 -> Meta
4 -> OpenAI


In [25]:
train_texts, val_texts, y_train, y_val = train_test_split(
    df["text"].tolist(),
    df["label_id"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label_id"]
)

print("Training samples:", len(train_texts))
print("Validation samples:", len(val_texts))

Training samples: 76941
Validation samples: 19236


In [26]:
MODEL_NAME = "distilbert-base-uncased"


In [27]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [28]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding=False
        )

        item = {key: torch.tensor(val) for key, val in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [29]:
MAX_LENGTH = 256   # try 384 later if texts are long and GPU is ok

train_dataset = TextDataset(train_texts, y_train, tokenizer, max_length=MAX_LENGTH)
val_dataset = TextDataset(val_texts, y_val, tokenizer, max_length=MAX_LENGTH)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [30]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")

    return {
        "accuracy": acc,
        "f1_macro": f1
    }

In [31]:
id2label = {i: label for i, label in enumerate(encoder.classes_)}
label2id = {label: i for i, label in enumerate(encoder.classes_)}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(encoder.classes_),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [32]:
training_args = TrainingArguments(
    output_dir="results/bert_run",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [33]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [34]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.216689,0.237133,0.908505,0.921027
2,0.164743,0.253142,0.923425,0.934537
3,0.090843,0.343537,0.921085,0.931321


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=14427, training_loss=0.2113853520561076, metrics={'train_runtime': 1411.7681, 'train_samples_per_second': 163.499, 'train_steps_per_second': 10.219, 'total_flos': 9292405985585250.0, 'train_loss': 0.2113853520561076, 'epoch': 3.0})

In [35]:
eval_results = trainer.evaluate()
print(eval_results)

{'eval_loss': 0.25326773524284363, 'eval_accuracy': 0.9232688708671242, 'eval_f1_macro': 0.9344184803568357, 'eval_runtime': 32.7037, 'eval_samples_per_second': 588.191, 'eval_steps_per_second': 36.785, 'epoch': 3.0}


In [41]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [36]:
# Load the professor dataset
df_prof = pd.read_csv("/content/subm1_labels_revealed.csv", sep=";", engine="python")

print(df_prof.head())
print(df_prof.columns.tolist())
print(df_prof.shape)

     ID                                               Text      Label
0  D2-1  A covalent bond is a chemical bond that involv...      Human
1  D2-2  A covalent bond forms when two atoms share one...  Anthropic
2  D2-3  A covalent bond is a type of chemical bond whe...     OpenAI
3  D2-4  A covalent bond is a chemical bond that involv...       Meta
4  D2-5  Driven by exciting developments in the field o...      Human
['ID', 'Text', 'Label']
(100, 3)


In [37]:
# Normalize the text and label columns
df_prof["Text"] = df_prof["Text"].astype(str).fillna("").str.strip()
df_prof["Label"] = df_prof["Label"].astype(str).fillna("").str.strip()

print(df_prof["Label"].value_counts())

Label
Human        34
Meta         18
Anthropic    17
Google       17
OpenAI       14
Name: count, dtype: int64


In [38]:
class TextOnlyDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=256):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding=False
        )

        return {key: torch.tensor(val) for key, val in encoding.items()}

prof_dataset = TextOnlyDataset(
    df_prof["Text"].tolist(),
    tokenizer,
    max_length=MAX_LENGTH
)

print("Professor test samples:", len(prof_dataset))

Professor test samples: 100


In [39]:
# Predict on the professor dataset
prof_predictions = trainer.predict(prof_dataset)

pred_ids = np.argmax(prof_predictions.predictions, axis=1)
pred_labels = encoder.inverse_transform(pred_ids)

df_prof["Predicted_Label"] = pred_labels

print(df_prof[["ID", "Text", "Label", "Predicted_Label"]].head())

     ID                                               Text      Label  \
0  D2-1  A covalent bond is a chemical bond that involv...      Human   
1  D2-2  A covalent bond forms when two atoms share one...  Anthropic   
2  D2-3  A covalent bond is a type of chemical bond whe...     OpenAI   
3  D2-4  A covalent bond is a chemical bond that involv...       Meta   
4  D2-5  Driven by exciting developments in the field o...      Human   

  Predicted_Label  
0           Human  
1       Anthropic  
2          OpenAI  
3          OpenAI  
4           Human  


In [42]:
# Evaluate the model on the professor dataset
y_true = df_prof["Label"].tolist()
y_pred = df_prof["Predicted_Label"].tolist()

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

print(f"Professor dataset accuracy: {acc:.4f}")
print(f"Professor dataset macro F1: {macro_f1:.4f}")

print("\nClassification report:\n")
print(classification_report(y_true, y_pred))

print("\nConfusion matrix:\n")
print(confusion_matrix(y_true, y_pred))

Professor dataset accuracy: 0.4400
Professor dataset macro F1: 0.3741

Classification report:

              precision    recall  f1-score   support

   Anthropic       0.22      0.24      0.23        17
      Google       0.44      0.65      0.52        17
       Human       0.74      0.68      0.71        34
        Meta       1.00      0.11      0.20        18
      OpenAI       0.17      0.29      0.21        14

    accuracy                           0.44       100
   macro avg       0.51      0.39      0.37       100
weighted avg       0.57      0.44      0.43       100


Confusion matrix:

[[ 4  8  4  0  1]
 [ 2 11  2  0  2]
 [ 3  2 23  0  6]
 [ 5  0  0  2 11]
 [ 4  4  2  0  4]]


In [44]:
df_sub = pd.read_csv("/content/subm2.csv", sep=";", engine="python")

In [45]:
possible_id_columns = ["ID", "Id", "id"]
possible_text_columns = ["Text", "text", "sentence", "Sentence", "content", "Content"]

id_column = None
text_column = None

for col in possible_id_columns:
    if col in df_sub.columns:
        id_column = col
        break

for col in possible_text_columns:
    if col in df_sub.columns:
        text_column = col
        break

print("ID column:", id_column)
print("Text column:", text_column)

if id_column is None:
    raise ValueError("No ID column found in submission file.")

if text_column is None:
    raise ValueError("No text column found in submission file.")

ID column: ID
Text column: Text


In [46]:
df_sub[text_column] = df_sub[text_column].astype(str).fillna("").str.strip()

print(df_sub[[id_column, text_column]].head())

       ID                                               Text
0  D2-101  Microbial mats of coexisting bacteria and arch...
1  D2-102  The origin of life on Earth remains a complex ...
2  D2-103  Estimates of the time at which life arose on E...
3  D2-104  Life on Earth emerged roughly 3.8-4 billion ye...
4  D2-105  Black holes predominantly form from the catast...


In [47]:
class TextOnlyDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=256):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding=False
        )
        return {key: torch.tensor(val) for key, val in encoding.items()}

In [48]:
sub_dataset = TextOnlyDataset(
    df_sub[text_column].tolist(),
    tokenizer,
    max_length=MAX_LENGTH
)

print("Submission samples:", len(sub_dataset))

Submission samples: 150


In [49]:
sub_predictions = trainer.predict(sub_dataset)

pred_ids = np.argmax(sub_predictions.predictions, axis=1)
pred_labels = encoder.inverse_transform(pred_ids)

df_sub["Label"] = pred_labels

print(df_sub[[id_column, text_column, "Label"]].head())

       ID                                               Text   Label
0  D2-101  Microbial mats of coexisting bacteria and arch...   Human
1  D2-102  The origin of life on Earth remains a complex ...    Meta
2  D2-103  Estimates of the time at which life arose on E...  OpenAI
3  D2-104  Life on Earth emerged roughly 3.8-4 billion ye...  Google
4  D2-105  Black holes predominantly form from the catast...  Google


In [50]:
submission = df_sub[[id_column, "Label"]].copy()
submission.to_csv("submission_transformer.csv", index=False)

print("Saved: submission_transformer.csv")
print(submission.head())

Saved: submission_transformer.csv
       ID   Label
0  D2-101   Human
1  D2-102    Meta
2  D2-103  OpenAI
3  D2-104  Google
4  D2-105  Google
